# MetroPT-3 TP2 regression and anomaly score baseline

This notebook evaluates a LightGBM model for `TP2` using chronological cross-validation and a later untouched test period. It calibrates the initial error threshold from out-of-fold (OOF) predictions. The raw CSV is not changed.

## 1. Load and prepare the data

The raw CSV is not changed. Continuous sensor values are averaged within non-overlapping 5-minute bins. Binary state signals use the majority state while preserving missing bins as missing. Bins with no sensor observations are dropped, and observed bins stay ordered by timestamp.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent
CSV_PATH = ROOT / 'data' / 'raw' / 'MetroPT3(AirCompressor).csv'

raw_df = pd.read_csv(CSV_PATH)
timestamp_col = next(col for col in raw_df.columns if col.lower() == 'timestamp')
raw_df[timestamp_col] = pd.to_datetime(raw_df[timestamp_col], errors='coerce')
raw_df = raw_df.dropna(subset=[timestamp_col]).sort_values(timestamp_col)

state_cols = [
    'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS',
    'Pressure_switch', 'Oil_level', 'Caudal_impulses'
]
signal_cols = [col for col in raw_df.select_dtypes(include='number').columns if col != 'Unnamed: 0']
# Aggregate each non-overlapping 5-minute bin; the raw CSV remains unchanged.
df = raw_df.set_index(timestamp_col)[signal_cols].resample('5min').mean()
for col in state_cols:
    if col in df.columns:
        df[col] = np.where(df[col].notna(), (df[col] >= 0.5).astype(float), np.nan)
df = df.dropna(how='all').reset_index().sort_values(timestamp_col).reset_index(drop=True)

print('Raw rows:', f'{len(raw_df):,}')
print('Observed 5-minute bins:', f'{len(df):,}')
print('Time range:', df[timestamp_col].min(), 'to', df[timestamp_col].max())
print('Median interval:', df[timestamp_col].diff().median())

## 2. Select the normal training period and untouched test period

The earlier exploration found `COMP=1` and `DV_eletric=0` in the common offloaded/off state. To avoid shutdown bins, this selection also requires 5-minute mean `Motor_current` between 3 and 5.5 A, around UCI's described offloaded current of 4 A.

February 1–20 is the training period. February 21–28 is held out as the final test period. Both use the same fixed active-offloaded filter. The test period is not used to select features, fit preprocessing, choose parameters, or set the error threshold.

In [ ]:
target = 'TP2'
features = [
    'TP3', 'H1', 'DV_pressure', 'Reservoirs',
    'Motor_current', 'COMP', 'DV_eletric', 'Towers'
]

train_start = pd.Timestamp('2020-02-01 00:00:00')
train_end = pd.Timestamp('2020-02-21 00:00:00')  # exclusive
test_start = train_end
test_end = pd.Timestamp('2020-03-01 00:00:00')  # exclusive

normal_mode = (
    df['COMP'].eq(1)
    & df['DV_eletric'].eq(0)
    & df['Motor_current'].between(3.0, 5.5)
)
train_df = df.loc[df[timestamp_col].ge(train_start) & df[timestamp_col].lt(train_end) & normal_mode].copy()
test_df = df.loc[df[timestamp_col].ge(test_start) & df[timestamp_col].lt(test_end) & normal_mode].copy()

print('Training range:', train_df[timestamp_col].min(), 'to', train_df[timestamp_col].max())
print('Training rows:', f'{len(train_df):,}', f'({len(train_df) / len(df):.2%} of all bins)')
print('Test range:', test_df[timestamp_col].min(), 'to', test_df[timestamp_col].max())
print('Test rows:', f'{len(test_df):,}', f'({len(test_df) / len(df):.2%} of all bins)')
print('Time ranges overlap:', train_df[timestamp_col].max() >= test_df[timestamp_col].min())

In [ ]:
# Show the training values and mark the untouched test window without plotting its target values.
fig_periods = go.Figure()
fig_periods.add_trace(go.Scatter(
    x=train_df[timestamp_col], y=train_df[target], mode='lines', name='Training period TP2'
))
fig_periods.add_vrect(
    x0=test_start, x1=test_end, fillcolor='gray', opacity=0.18, line_width=0,
    annotation_text='Untouched test window', annotation_position='top left'
)
fig_periods.update_layout(
    title='Training data and reserved test window', xaxis_title='Timestamp', yaxis_title='TP2',
    hovermode='x', template='plotly_white', dragmode='pan'
)
fig_periods.show(config={'scrollZoom': True})

## 3. Select target and inspect the fixed features

The target is `TP2`. I keep the previously selected pressure, current, and state signals. The timestamp is used only for ordering and plots. Feature diagnostics below use the selected training rows only; they do not change the predefined feature list.

In [ ]:
feature_check = pd.DataFrame(index=features)
feature_check['data type'] = train_df[features].dtypes.astype(str)
feature_check['missing values'] = train_df[features].isna().sum()
feature_check['unique values'] = train_df[features].nunique(dropna=False)
feature_check['standard deviation'] = train_df[features].std()
feature_check['near-constant'] = [
    train_df[col].nunique(dropna=False) <= 1
    or train_df[col].std() <= 1e-6 * max(1, abs(train_df[col].mean()))
    for col in features
]
feature_check['correlation with TP2'] = [
    train_df[col].corr(train_df[target]) if train_df[col].nunique(dropna=True) > 1 else np.nan
    for col in features
]
feature_check['predictor role'] = [
    'Pneumatic panel pressure.',
    'Pressure signal in the compressor air system.',
    'Air-dryer discharge pressure behavior.',
    'Downstream pressure related to TP3.',
    'Motor load and operating-state information.',
    'Compressor intake valve state.',
    'Compressor outlet valve state.',
    'Air-drying tower state.'
]
display(feature_check)
print('Features:', features)
print('Training feature missing values:', int(train_df[features].isna().sum().sum()))

## 4. Prepare the chronological splits

All rows are ordered by timestamp. The final February test period stays separate. Imputation medians are learned inside each CV fold; for the final model they are learned from the training period only. The current dataset has no missing values in the selected features.

In [ ]:
X_train = train_df[features].copy().reset_index(drop=True)
y_train = train_df[target].copy().reset_index(drop=True)
train_timestamps = train_df[timestamp_col].copy().reset_index(drop=True)
X_test = test_df[features].copy().reset_index(drop=True)
y_test = test_df[target].copy().reset_index(drop=True)
test_timestamps = test_df[timestamp_col].copy().reset_index(drop=True)

if y_train.isna().any() or y_test.isna().any():
    raise ValueError('Target TP2 contains missing values in the selected periods.')

print('Training bins:', len(X_train))
print('Reserved test bins:', len(X_test))
print('Training features with missing values:', int(X_train.isna().sum().sum()))

## 5. Time-series cross-validation

The data is time-dependent, so I use five expanding-window `TimeSeriesSplit` folds instead of shuffled K-fold. Each validation fold occurs after its fold's training rows. The earliest warm-up block has no OOF prediction because there is no earlier training data for it; OOF metrics and residuals use all later rows that were validated out of sample.

In [ ]:
model_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'n_jobs': 1,
    'verbosity': -1
}
cv = TimeSeriesSplit(n_splits=5)
oof_pred = np.full(len(X_train), np.nan)
fold_number = np.full(len(X_train), -1, dtype=int)
fold_rows = []

for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_train), start=1):
    X_fit = X_train.iloc[fit_idx].copy()
    X_valid = X_train.iloc[valid_idx].copy()
    y_fit = y_train.iloc[fit_idx]
    y_valid = y_train.iloc[valid_idx]

    # Fit imputation values on this fold's training rows only.
    fold_medians = X_fit.median(numeric_only=True)
    X_fit = X_fit.fillna(fold_medians)
    X_valid = X_valid.fillna(fold_medians)

    fold_model = LGBMRegressor(**model_params)
    fold_model.fit(X_fit, y_fit)
    fold_pred = fold_model.predict(X_valid)
    oof_pred[valid_idx] = fold_pred
    fold_number[valid_idx] = fold

    fold_rows.append({
        'Fold': fold,
        'Training rows': len(fit_idx),
        'Validation rows': len(valid_idx),
        'MAE': mean_absolute_error(y_valid, fold_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid, fold_pred)),
        'R²': r2_score(y_valid, fold_pred)
    })

fold_metrics = pd.DataFrame(fold_rows)
display(fold_metrics)
cv_fold_summary = fold_metrics[['MAE', 'RMSE', 'R²']].agg(['mean', 'std']).T
cv_fold_summary.columns = ['Fold mean', 'Fold standard deviation']
print('Fold performance (mean ± standard deviation):')
display(cv_fold_summary)

oof_mask = np.isfinite(oof_pred)
oof_residual = y_train.to_numpy()[oof_mask] - oof_pred[oof_mask]
oof_abs_residual = np.abs(oof_residual)
oof_metrics = {
    'MAE': mean_absolute_error(y_train.to_numpy()[oof_mask], oof_pred[oof_mask]),
    'RMSE': np.sqrt(mean_squared_error(y_train.to_numpy()[oof_mask], oof_pred[oof_mask])),
    'R²': r2_score(y_train.to_numpy()[oof_mask], oof_pred[oof_mask])
}
oof_results = pd.DataFrame({
    'Timestamp': train_timestamps.to_numpy()[oof_mask],
    'Actual TP2': y_train.to_numpy()[oof_mask],
    'Predicted TP2': oof_pred[oof_mask],
    'Residual': oof_residual,
    'Absolute residual': oof_abs_residual,
    'Fold': fold_number[oof_mask]
})
print('OOF rows:', len(oof_results), '| warm-up rows without OOF prediction:', int((~oof_mask).sum()))
print('Pooled OOF metrics:', oof_metrics)

## 6. Fit the final model and evaluate the reserved test period

After CV, fit the fixed LightGBM configuration once on the full training period. Only now predict and evaluate the reserved February 21–28 test period. The test set is not used for model or threshold selection.

In [ ]:
# Final imputation values use training rows only.
train_medians = X_train.median(numeric_only=True)
X_train_final = X_train.fillna(train_medians)
X_test_final = X_test.fillna(train_medians)

model = LGBMRegressor(**model_params)
model.fit(X_train_final, y_train)
train_pred = model.predict(X_train_final)
y_pred = model.predict(X_test_final)  # one final test prediction

train_metrics = {
    'MAE': mean_absolute_error(y_train, train_pred),
    'RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
    'R²': r2_score(y_train, train_pred)
}
test_metrics = {
    'MAE': mean_absolute_error(y_test, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
    'R²': r2_score(y_test, y_pred)
}
performance = pd.DataFrame([
    {'Split': 'Train (in-sample)', **train_metrics},
    {'Split': 'OOF (pooled)', **oof_metrics},
    {'Split': 'Test (final)', **test_metrics}
]).set_index('Split')
display(performance)

generalization_gaps = pd.DataFrame({
    'MAE gap (split - train)': [oof_metrics['MAE'] - train_metrics['MAE'], test_metrics['MAE'] - train_metrics['MAE']],
    'RMSE gap (split - train)': [oof_metrics['RMSE'] - train_metrics['RMSE'], test_metrics['RMSE'] - train_metrics['RMSE']],
    'R² gap (train - split)': [train_metrics['R²'] - oof_metrics['R²'], train_metrics['R²'] - test_metrics['R²']]
}, index=['OOF', 'Test'])
print('Generalization gaps (positive means split error is higher, or train R² is higher):')
display(generalization_gaps)

In [ ]:
comparison = pd.DataFrame({'Actual TP2': y_test.to_numpy(), 'Predicted TP2': y_pred})
fig_scatter = px.scatter(
    comparison, x='Actual TP2', y='Predicted TP2', opacity=0.6,
    title='Final test: actual vs predicted TP2', template='plotly_white'
)
low = min(comparison.min())
high = max(comparison.max())
fig_scatter.add_trace(go.Scatter(x=[low, high], y=[low, high], mode='lines', name='Perfect prediction'))
fig_scatter.update_layout(dragmode='pan')
fig_scatter.show(config={'scrollZoom': True})

test_prediction_plot = pd.DataFrame({
    'Timestamp': test_timestamps.to_numpy(),
    'Actual TP2': y_test.to_numpy(),
    'Predicted TP2': y_pred
})
fig_test_time = go.Figure()
fig_test_time.add_trace(go.Scatter(x=test_prediction_plot['Timestamp'], y=test_prediction_plot['Actual TP2'], mode='lines', name='Actual TP2'))
fig_test_time.add_trace(go.Scatter(x=test_prediction_plot['Timestamp'], y=test_prediction_plot['Predicted TP2'], mode='lines', name='Predicted TP2'))
fig_test_time.update_layout(
    title='Final test: actual and predicted TP2', xaxis_title='Timestamp', yaxis_title='TP2',
    hovermode='x unified', template='plotly_white', dragmode='pan'
)
fig_test_time.show(config={'scrollZoom': True})

## 7. Out-of-sample residual analysis

OOF residuals are the primary residual evidence here. The test residuals are also summarized after the one final test evaluation. Neither set is used to refit the model.

In [ ]:
oof_residual_stats = pd.Series({
    'Mean residual / bias': oof_residual.mean(),
    'MAE': mean_absolute_error(y_train.to_numpy()[oof_mask], oof_pred[oof_mask]),
    'Residual standard deviation': oof_residual.std(ddof=1),
    '95% absolute residual': np.quantile(oof_abs_residual, 0.95),
    '97.5% absolute residual': np.quantile(oof_abs_residual, 0.975),
    '99% absolute residual': np.quantile(oof_abs_residual, 0.99),
    '99.5% absolute residual': np.quantile(oof_abs_residual, 0.995),
}, name='OOF value')
test_residual = y_test.to_numpy() - y_pred
test_abs_residual = np.abs(test_residual)
test_residual_stats = pd.Series({
    'Mean residual / bias': test_residual.mean(),
    'MAE': mean_absolute_error(y_test, y_pred),
    'Residual standard deviation': test_residual.std(ddof=1),
    '95% absolute residual': np.quantile(test_abs_residual, 0.95),
    '97.5% absolute residual': np.quantile(test_abs_residual, 0.975),
    '99% absolute residual': np.quantile(test_abs_residual, 0.99),
    '99.5% absolute residual': np.quantile(test_abs_residual, 0.995),
}, name='Test value')
display(pd.concat([oof_residual_stats, test_residual_stats], axis=1))

In [ ]:
fig_oof_residuals = px.histogram(
    x=oof_residual, nbins=60, labels={'x': 'OOF residual'},
    title='Out-of-fold residual distribution', template='plotly_white'
)
fig_oof_residuals.add_vline(x=0, line_dash='dash', line_color='black')
fig_oof_residuals.update_layout(dragmode='pan')
fig_oof_residuals.show(config={'scrollZoom': True})

fig_resid_pred = px.scatter(
    x=oof_results['Predicted TP2'], y=oof_results['Residual'], opacity=0.6,
    labels={'x': 'OOF predicted TP2', 'y': 'OOF residual'},
    title='OOF residuals vs predicted TP2', template='plotly_white'
)
fig_resid_pred.add_hline(y=0, line_dash='dash', line_color='black')
fig_resid_pred.update_layout(dragmode='pan')
fig_resid_pred.show(config={'scrollZoom': True})

## 8. Calibrate an initial error threshold

The main threshold is the 99th percentile of absolute OOF residuals. These errors come from time-forward validation rows whose fold models did not train on them, so this is a more defensible calibration than fitted training residuals. It means about 1% of the pooled OOF absolute errors exceed the threshold by construction; it does not mean 1% of future rows must be above it.

For comparison, I also calculate OOF mean absolute error plus three standard deviations of OOF absolute errors. The two approaches summarize different parts and shapes of the error distribution; the quantile is the selected baseline. The final test set is not used to choose either value.

In [ ]:
threshold_99 = np.quantile(oof_abs_residual, 0.99)
oof_mean_abs_error = oof_abs_residual.mean()
oof_std_abs_error = oof_abs_residual.std(ddof=1)
threshold_mean_plus_3std = oof_mean_abs_error + 3 * oof_std_abs_error

threshold_comparison = pd.DataFrame({
    'Method': ['99th percentile of absolute OOF residuals', 'OOF mean absolute residual + 3 × OOF standard deviation'],
    'Threshold': [threshold_99, threshold_mean_plus_3std]
})
display(threshold_comparison)
print(f'Selected baseline threshold: {threshold_99:.6f} TP2 units')

## 9. Score every non-empty 5-minute bin

In [ ]:
X_all = df[features].copy().fillna(train_medians)
y_all_pred = model.predict(X_all)
residual_all = df[target].to_numpy() - y_all_pred
abs_residual = np.abs(residual_all)
raw_anomaly_score = pd.Series(abs_residual / threshold_99, index=df.index)
anomaly_score = raw_anomaly_score.clip(upper=1.0)
is_anomaly = raw_anomaly_score >= 1

all_results = pd.DataFrame({
    'Timestamp': df[timestamp_col].to_numpy(),
    'Actual TP2': df[target].to_numpy(),
    'Predicted TP2': y_all_pred,
    'Residual': residual_all,
    'Absolute residual': abs_residual,
    'raw_anomaly_score': raw_anomaly_score,
    'anomaly_score': anomaly_score,
    'is_anomaly': is_anomaly
})
print('Scored bins:', len(all_results), '| Resampled bins:', len(df))
print('All bins have scores:', all_results['anomaly_score'].notna().all())
display(all_results.head())

## 10. Visualize scores and predictions

In [ ]:
fig_score = go.Figure()
fig_score.add_trace(go.Scattergl(
    x=all_results['Timestamp'], y=all_results['anomaly_score'],
    mode='lines', name='Anomaly score'
))
fig_score.add_hline(y=1.0, line_dash='dash', line_color='red', line_width=2, annotation_text='Threshold = 1.0')
fig_score.update_layout(
    title='Clipped anomaly score over time', xaxis_title='Timestamp', yaxis_title='Anomaly score (0–1)',
    hovermode='x', template='plotly_white', dragmode='pan',
    yaxis=dict(range=[0, 1])
)
fig_score.show(config={'scrollZoom': True})

fig_all = go.Figure()
fig_all.add_trace(go.Scattergl(x=all_results['Timestamp'], y=all_results['Actual TP2'], mode='lines', name='Actual TP2'))
fig_all.add_trace(go.Scattergl(x=all_results['Timestamp'], y=all_results['Predicted TP2'], mode='lines', name='Predicted TP2'))
above = all_results['is_anomaly']
fig_all.add_trace(go.Scattergl(
    x=all_results.loc[above, 'Timestamp'], y=all_results.loc[above, 'Actual TP2'],
    mode='markers', name='Above threshold', marker=dict(color='red', size=5)
))
fig_all.update_layout(
    title='Actual and predicted TP2 with above-threshold points', xaxis_title='Timestamp',
    yaxis_title='TP2', hovermode='x unified', template='plotly_white', dragmode='pan'
)
fig_all.show(config={'scrollZoom': True})

## 11. Detected points and consecutive runs

In [ ]:
total_bins = len(all_results)
detected_count = int(all_results['is_anomaly'].sum())
detected_percent = 100 * detected_count / total_bins if total_bins else 0
max_raw_score = all_results['raw_anomaly_score'].max()
max_score = all_results['anomaly_score'].max()
score_summary = pd.DataFrame({
    'Value': [total_bins, detected_count, detected_percent, max_raw_score, max_score]
}, index=['Total 5-minute bins', 'Above-threshold bins', 'Above-threshold percentage (%)', 'Maximum raw anomaly score', 'Maximum clipped anomaly score'])
display(score_summary)

top_anomalies = all_results.nlargest(20, 'raw_anomaly_score')[
    ['Timestamp', 'Actual TP2', 'Predicted TP2', 'Residual', 'Absolute residual', 'raw_anomaly_score', 'anomaly_score']
]
display(top_anomalies)

# A gap in the resampled timeline starts a new run.
time_gap = all_results['Timestamp'].diff().gt(pd.Timedelta('5min'))
run_id = (all_results['is_anomaly'].ne(all_results['is_anomaly'].shift()) | time_gap).cumsum()
runs = all_results.groupby(run_id, sort=False).agg(
    is_anomaly=('is_anomaly', 'first'), start=('Timestamp', 'first'),
    end=('Timestamp', 'last'), bins=('Timestamp', 'size')
)
anomaly_runs = runs.loc[runs['is_anomaly']].copy()
run_length_counts = anomaly_runs.groupby('bins').size().rename('run count').to_frame().sort_index()
print('Above-threshold runs:', len(anomaly_runs))
print('Run-length distribution (5-minute bins):')
display(run_length_counts)
print('Longest runs:')
display(anomaly_runs.nlargest(20, 'bins'))

## 12. Leakage and validation checks

- The final test dates do not overlap the training dates; the boundary is February 21.
- Time-series CV uses only earlier rows to predict each later fold. The initial warm-up rows have no OOF predictions.
- Feature diagnostics and imputation values are based on training rows; CV imputation is refit within each fold.
- The feature list is fixed in advance. The timestamp and row identifier are not predictors.
- No random split, test-driven tuning, test-based threshold calibration, or data removal is used.
- Resampling averages readings within separate 5-minute bins before the date split. It does not share bins across the train/test boundary. Predictors and target are same-bin measurements, so this estimates contemporaneous TP2 rather than forecasting future TP2.
- The final model is trained only on active offloaded bins. Scores in other operating states are distribution shifts and can be high even during valid operation.
- The threshold is based on OOF residuals from the selected February training period. The test residuals are reported for evaluation only.

## 13. Final performance report

In [ ]:
cv_mean = cv_fold_summary['Fold mean']
cv_std = cv_fold_summary['Fold standard deviation']
mae_gap_cv = oof_metrics['MAE'] - train_metrics['MAE']
mae_gap_test = test_metrics['MAE'] - train_metrics['MAE']
print('MODEL PERFORMANCE')
print('-----------------')
print(f"Train: MAE={train_metrics['MAE']:.6f}, RMSE={train_metrics['RMSE']:.6f}, R²={train_metrics['R²']:.6f}")
print(f"CV folds (mean ± std): MAE={cv_mean['MAE']:.6f} ± {cv_std['MAE']:.6f}, RMSE={cv_mean['RMSE']:.6f} ± {cv_std['RMSE']:.6f}, R²={cv_mean['R²']:.6f} ± {cv_std['R²']:.6f}")
print(f"Pooled OOF: MAE={oof_metrics['MAE']:.6f}, RMSE={oof_metrics['RMSE']:.6f}, R²={oof_metrics['R²']:.6f}")
print(f"Test: MAE={test_metrics['MAE']:.6f}, RMSE={test_metrics['RMSE']:.6f}, R²={test_metrics['R²']:.6f}")
print('\nGENERALIZATION')
print('--------------')
print(f"OOF minus train MAE gap: {mae_gap_cv:.6f}; test minus train MAE gap: {mae_gap_test:.6f}")
print('Generalization assessment: Test MAE and RMSE are within one fold standard deviation of the CV means; the later test period is consistent with CV, though evidence is limited to one regime and short dates.')
print('Overfitting assessment: Train errors are substantially lower than OOF errors, indicating some overfit despite strong time-forward and test R².')
print('\nRESIDUALS')
print('---------')
print(f"OOF residual mean={oof_residual.mean():.6f}, std={oof_residual.std(ddof=1):.6f}")
print(f"OOF absolute residual q95={np.quantile(oof_abs_residual, .95):.6f}, q97.5={np.quantile(oof_abs_residual, .975):.6f}, q99={np.quantile(oof_abs_residual, .99):.6f}, q99.5={np.quantile(oof_abs_residual, .995):.6f}")
print(f"Test residual mean={test_residual.mean():.6f}, std={test_residual.std(ddof=1):.6f}")
print('\nTHRESHOLD')
print('---------')
print(f"Recommended initial threshold: {threshold_99:.6f} (99th percentile of OOF absolute residuals)")
print(f"Comparison: OOF mean absolute residual + 3 standard deviations = {threshold_mean_plus_3std:.6f}")
print('Threshold confidence: limited to moderate; the 99th percentile uses 950 serially dependent OOF bins from one short period and one selected mode.')
print('\nFINAL CONCLUSION')
print('----------------')
print('The model predicts TP2 accurately within the selected active-offloaded regime on this short later test period, but there is some train-to-OOF overfit. There is not yet enough evidence for operational anomaly decisions: validate longer periods and other operating modes. Do not interpret high scores in other modes as failures.')

## Conclusion

Use the printed performance report together with the fold table, OOF residual plots, and final test metrics. A strong in-sample score alone is not evidence of generalization. This baseline applies to the selected active-offloaded regime and contemporaneous TP2 estimation; broader validation is needed before operational use. Above-threshold scores are review signals, not confirmed failures.